# Academic Multi-Agent Deep Researcher

A capstone LangGraph project that combines every pattern from Weeks 4 labs:

| Pattern | Lab |
|---------|-----|
| 5-step graph (State → Compile) | Lab 1 |
| `operator.add` reducer for parallel results | Lab 1 |
| ToolNode + tool binding | Lab 2 |
| SqliteSaver persistent memory | Lab 2 |
| Async Gradio handlers + live progress via `graph.stream` (sync API; SqliteSaver is not async) | Lab 3 |
| Structured outputs (Pydantic) | Lab 4 |
| `interrupt_after` human-in-the-loop | Exercise 9 |
| `Send` parallel fan-out | Exercise 10 |

**Flow:**
```
START → clarifier [interrupt] → task_planner
  → [agent_discovery | agent_fulltext | agent_context | agent_domain] (parallel)
  → synthesizer → rater → report_writer → email_agent → END
```

**UI:** Gradio chat shows clarifying questions, the refined research question after you answer, and step-by-step pipeline status. The full report is markdown on the page; email uses SendGrid (HTML converted from markdown).

**Modules used:**
- `schemas.py` — State TypedDict and all Pydantic output models
- `tools.py` — External API wrappers (Semantic Scholar, ArXiv, CORE, Tavily, PubMed, SendGrid email)


## Step 1 — Install Dependencies

Environments created with **`uv`** often have no `pip` module, so **`%pip install` fails**. Use the **next** cell (`uv pip install …`) instead. After installing or upgrading packages, use **Kernel → Restart** if imports still fail.


In [ ]:
!uv pip install -q langgraph langchain langchain-openai langchain-community langgraph-checkpoint-sqlite
!uv pip install -q arxiv                  # ArXiv preprint search
!uv pip install -q langchain-tavily       # Tavily web search (recommended)
!uv pip install -q biopython              # PubMed via Entrez
!uv pip install -q markdown               # Markdown → HTML for email
!uv pip install -q sendgrid               # SendGrid API email
!uv pip install -q gradio                 # UI
!uv pip install -q aiosqlite              # required if you use graph.ainvoke with SQLite checkpoint

import importlib.util

if importlib.util.find_spec("aiosqlite") is not None:
    import aiosqlite  # noqa: F401
    print("aiosqlite: import OK")
else:
    print("aiosqlite: not installed (OK here: Gradio uses sync graph.invoke via asyncio.to_thread)")



## Step 2 — Import Libraries

In [ ]:
# Load .env FIRST — must happen before LangChain imports so the tracer picks up env vars
import os
from dotenv import load_dotenv
load_dotenv(override=True)

# Standard library and third-party imports
import asyncio
import contextvars
import time
import re
import uuid
import sqlite3
import threading
from typing import List

import gradio as gr
from IPython.display import Image, display

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langgraph.checkpoint.sqlite import SqliteSaver

## Step 3 — Load Environment Variables and Verify API Keys

In [ ]:
required = [
    "OPENAI_API_KEY",
    "TAVILY_API_KEY",
    "SENDGRID_API_KEY",
    "CORE_API_KEY",
    "EMAIL_FROM",
    "LANGSMITH_API_KEY",
    "LANGSMITH_PROJECT",
    "LANGSMITH_TRACING",
    "LANGSMITH_ENDPOINT",
]

for key in required:
    status = "OK" if os.getenv(key) else "MISSING"
    print(f"  {key}: {status}")

print(f"  LANGSMITH_PROJECT: {os.getenv('LANGSMITH_PROJECT', '—')}")


## Step 4 — Import Schemas and Tools from Modules

In [ ]:
# Import data models and State from schemas.py
# Import all API wrapper functions from tools.py
from schemas import State, PlannerOutput, RaterOutput, FinalReport
from tools import (
    search_semantic_scholar,
    search_arxiv,
    search_core,
    search_tavily,
    search_pubmed,
    send_email_report,
)

print("Schemas and tools loaded.")


## Step 5 — Initialise Language Models

Four LLM variants, each with a specific role:
- `llm` — base model for clarifier and synthesizer (plain text output)
- `planner_llm` — structured output → `PlannerOutput`
- `rater_llm` — structured output → `RaterOutput`
- `writer_llm` — structured output → `FinalReport`

In [ ]:
DEFAULT_MODEL = "gpt-4o-mini"

PLANNER_MODEL = "gpt-4o-mini"
RATER_MODEL = "gpt-4o-mini"
WRITER_MODEL = "gpt-4o-mini"

# Initialise all LLM variants used across node functions
llm = ChatOpenAI(model=DEFAULT_MODEL, temperature=0.3)

planner_llm = ChatOpenAI(model=PLANNER_MODEL, temperature=0).with_structured_output(PlannerOutput)
rater_llm   = ChatOpenAI(model=RATER_MODEL, temperature=0).with_structured_output(RaterOutput)
writer_llm  = ChatOpenAI(model=WRITER_MODEL, temperature=0.4).with_structured_output(FinalReport)

print("LLMs initialised.")

## Step 6 — Initialise Persistent Memory (SqliteSaver)

Uses `SqliteSaver` from lab 2 so research sessions survive notebook restarts.
Gradio runs the graph with **`graph.invoke` inside `asyncio.to_thread`** so the sync checkpointer does not require `aiosqlite` (unlike `graph.ainvoke` with the same saver).
Each session is identified by a unique `thread_id`.

In [ ]:
# Set up SQLite checkpointer for persistent session memory
conn = sqlite3.connect("data/research_memory.db", check_same_thread=False)
memory = SqliteSaver(conn)

print("SqliteSaver memory initialised → research_memory.db")

---
## Node 1 — Clarifier
Generates 4 targeted clarifying questions about the research topic.
The graph **pauses after this node** (`interrupt_after`) for the user to answer.

In [ ]:
CLARIFIER_SYSTEM_PROMPT = """
You are a research assistant scoping a deep academic research request.
"""
CLARIFIER_HUMAN_PROMPT = """
The user wants deep research on: "{topic}"


Generate exactly 4 clarifying questions to better understand:
1. The specific angle or sub-topic they are interested in
2. The technical depth and intended audience
3. Any time period, geographic, or domain constraints
4. Preferred emphasis (theoretical foundations, practical applications, recent developments, etc.)

Return ONLY a numbered list of 4 questions. No preamble, no explanation.
"""

def clarifier_node(state: State) -> dict:
    """Generate clarifying questions about the research topic."""
    topic = state["topic"]

    response = llm.invoke([
        SystemMessage(content="You are a research assistant scoping a deep academic research request."),
        HumanMessage(content=CLARIFIER_HUMAN_PROMPT.format(topic=topic))
    ])

    lines = [l.strip() for l in response.content.split("\n") if l.strip()]
    questions = [re.sub(r"^\d+[\.)\s]+", "", line) for line in lines if line]
    questions = [q for q in questions if len(q) > 10][:4]

    print(f"[Clarifier] Generated {len(questions)} questions.")
    return {"clarifying_questions": questions}


print("Clarifier node loaded")

## Node 2 — Task Planner
Breaks the topic into research sub-tasks and generates one optimised query per specialist agent.
Uses `PlannerOutput` structured output to guarantee a clean `agent_queries` dict.

In [ ]:
TASK_PLANNER_SYSTEM_PROMPT = """
You are an expert academic research planner.
"""
TASK_PLANNER_HUMAN_PROMPT = """ 
    Topic: "{topic}"

    User clarifications:
    {qa_text}

    First produce refined_research_question: one clear, standalone research question that merges the topic and clarifications.

    Then create a research plan with sub-tasks and 3–4 search queries, one per agent:
    - semantic_scholar_query: formal academic terminology, paper-title style
    - arxiv_query: technical keywords and field-specific jargon
    - tavily_query: natural language for recent news and practical applications
    - pubmed_query + needs_pubmed=true ONLY if topic is medical/biological/clinical

    Each query must explore a meaningfully different angle of the topic.
        
    """

def task_planner_node(state: State) -> dict:
    """Break the research topic into tasks and assign one query per specialist agent."""
    topic = state["topic"]
    qa = state.get("clarifying_qa", [])

    qa_text = "\n".join(
        f"Q: {item['question']}\nA: {item['answer']}" for item in qa
    ) or "No clarifications provided."

    result = planner_llm.invoke([
        SystemMessage(content=TASK_PLANNER_SYSTEM_PROMPT),
        HumanMessage(content=TASK_PLANNER_HUMAN_PROMPT.format(topic=topic, qa_text=qa_text))
    ])

    print(f"[Task Planner] {len(result.research_tasks)} tasks | PubMed needed: {result.needs_pubmed}")
    return {
        "refined_topic": result.refined_research_question,
        "research_tasks": result.research_tasks,
        "agent_queries": result.model_dump(),
    }

print("Task planner node loaded")


## Fan-Out Router — Dispatch Queries to Parallel Agents via `Send`

Returns a list of `Send` objects — one per specialist agent.
LangGraph runs all of them in the same super-step (parallel execution).
Results accumulate in `raw_results` via the `operator.add` reducer.

In [ ]:
query_route_map = {
    "semantic_scholar_query": "agent_discovery",
    "arxiv_query": "agent_fulltext",
    "tavily_query": "agent_context",
    "pubmed_query": "agent_domain",
}

def route_queries(state: State) -> List[Send]:
    """Route each specialist agent its own query using the Send API."""
    queries = state.get("agent_queries", {})
    # topic = state["topic"]

    return [
        Send(query_route_map[k], {"query": v})
        for k, v in queries.items()
    ]

print("Query router loaded")

def fan_out_researchers(state: State) -> List[Send]:
    """Route each specialist agent its own query using the Send API."""
    queries = state.get("agent_queries", {})
    topic = state["topic"]

    return [
        Send("agent_discovery", {
            "query": queries.get("semantic_scholar_query", topic)
        }),
        Send("agent_fulltext", {
            "query": queries.get("arxiv_query", topic)
        }),
        Send("agent_context", {
            "query": queries.get("tavily_query", topic)
        }),
        Send("agent_domain", {
            "query": queries.get("pubmed_query") or topic,
            "needs_pubmed": queries.get("needs_pubmed", False),
        }),
    ]

## Node 3a — Discovery Agent (Semantic Scholar)
Searches for academic papers and returns titles, abstracts, citation counts,
and open-access URLs. Citation count feeds directly into the rater as a credibility signal.

In [ ]:
def agent_discovery(inputs: dict) -> dict:
    """Search Semantic Scholar for papers — returns citation metadata for the rater."""
    time.sleep(0.4)
    query = inputs.get("query", "")
    results = search_semantic_scholar(query)
    print(f"[Discovery Agent] {len(results)} papers from Semantic Scholar")
    return {"raw_results": results}


## Node 3b — Full-Text Agent (ArXiv + CORE)
Retrieves full abstract content from ArXiv (STEM preprints) and CORE (open-access,
all fields). Together they cover the full-text gap that Semantic Scholar alone leaves.

In [ ]:
def agent_fulltext(inputs: dict) -> dict:
    """Retrieve paper content from ArXiv (STEM) and CORE API (all fields)."""
    time.sleep(3.6)
    query = inputs.get("query", "")
    arxiv_results = search_arxiv(query)
    core_results = search_core(query)
    results = arxiv_results + core_results
    print(f"[Full-Text Agent] {len(arxiv_results)} ArXiv + {len(core_results)} CORE papers")
    return {"raw_results": results}


## Node 3c — Context Agent (Tavily)
Retrieves recent web content, news, and commentary that contextualises the academic findings.
Tavily fills the gap for content not yet in academic databases.

In [ ]:
def agent_context(inputs: dict) -> dict:
    """Search Tavily for recent web content and news around the topic."""
    time.sleep(1.2)
    query = inputs.get("query", "")
    results = search_tavily(query)
    print(f"[Context Agent] {len(results)} results from Tavily")
    return {"raw_results": results}


## Node 3d — Domain Agent (PubMed — conditional)
Searches PubMed for biomedical literature. Only activates when the task planner
sets `needs_pubmed=True`. Returns an empty list otherwise — the `operator.add`
reducer handles this gracefully.

In [ ]:
def agent_domain(inputs: dict) -> dict:
    """Search PubMed for biomedical papers — no-op if topic is not biomedical."""
    if not inputs.get("needs_pubmed", False):
        print("[Domain Agent] Skipped — topic is not biomedical.")
        return {"raw_results": []}

    time.sleep(0.5)
    query = inputs.get("query", "")
    results = search_pubmed(query)
    print(f"[Domain Agent] {len(results)} papers from PubMed")
    return {"raw_results": results}


## Node 4 — Synthesiser
Merges all parallel agent results into a coherent thematic synthesis.
Each source is truncated to 500 chars before the LLM call to avoid token overflow.

In [ ]:
def synthesizer_node(state: State) -> dict:
    """Merge all agent results into a structured thematic synthesis."""
    raw = state.get("raw_results", [])
    topic = state["topic"]

    if not raw:
        return {"synthesized_findings": "No research results were retrieved."}

    # Truncate each result to control token usage
    summaries = []
    for r in raw:
        source = r.get("source", "unknown").upper()
        title = r.get("title", "Untitled")
        content = (r.get("content") or "")[:500]
        url = r.get("url", "")
        cite = r.get("citation_count")
        cite_note = f" [cited {cite}x]" if cite else ""
        summaries.append(f"[{source}{cite_note}] {title}\n{content}\nURL: {url}")

    combined = "\n\n---\n\n".join(summaries)

    response = llm.invoke([
        SystemMessage(content="You are a senior research analyst synthesizing academic literature."),
        HumanMessage(content=(
            f'Synthesize these research results on: "{topic}"\n\n'
            "Group findings by theme. Highlight consensus across sources. "
            "Note any contradictions or gaps. Be concise (500–700 words).\n\n"
            f"{combined}"
        ))
    ])

    print(f"[Synthesiser] Synthesised {len(raw)} results.")
    return {"synthesized_findings": response.content}

## Node 5 — Rater
Scores each research point for confidence and relevance.
Uses `citation_count` from Semantic Scholar as a built-in credibility signal.
Filters out points below 0.4 on either dimension.

In [ ]:
def rater_node(state: State) -> dict:
    """Score each research finding for confidence and relevance using structured output."""
    raw = state.get("raw_results", [])
    topic = state["topic"]

    if not raw:
        return {"rated_points": []}

    # Build a compact representation — cap at 20 results to stay within token budget
    items = []
    for r in raw[:20]:
        items.append(
            f"Title: {r.get('title', '')}\n"
            f"Source: {r.get('source', '')} ({r.get('source_type', '')})\n"
            f"Citation count: {r.get('citation_count', 'N/A')}\n"
            f"Content: {(r.get('content') or '')[:300]}\n"
            f"URL: {r.get('url', '')}"
        )

    result = rater_llm.invoke([
        SystemMessage(content="You are a rigorous academic research evaluator."),
        HumanMessage(content=(
            f'Rate these research findings for the topic: "{topic}"\n\n'
            "Scoring rules for confidence_score:\n"
            "  peer-reviewed paper → up to 1.0\n"
            "  ArXiv preprint → up to 0.85\n"
            "  web/Tavily source → max 0.70\n"
            "  higher citation_count boosts score within the range\n"
            "Include ONLY points where confidence ≥ 0.4 AND relevance ≥ 0.4.\n\n"
            + "\n\n---\n\n".join(items)
        ))
    ])

    rated = [p.model_dump() for p in result.rated_points]
    print(f"[Rater] {len(rated)} points passed the quality threshold.")
    return {"rated_points": rated}

## Node 6 — Report Writer
Assembles the final structured report from the top-rated findings.
Uses `FinalReport` structured output to guarantee all sections are present.

In [ ]:
def report_writer_node(state: State) -> dict:
    """Assemble the final deep research report from rated findings."""
    topic = state["topic"]
    qa = state.get("clarifying_qa", [])
    rated = state.get("rated_points", [])
    synthesis = state.get("synthesized_findings", "")

    if not rated:
        print("[Report Writer] No rated points — writing from synthesis only.")

    qa_context = "\n".join(
        f"- {item['question']}: {item['answer']}" for item in qa
    ) or "No clarifications provided."

    # Sort by combined score and take top 15
    top_points = sorted(
        rated,
        key=lambda p: p.get("confidence_score", 0) * p.get("relevance_score", 0),
        reverse=True,
    )[:15]

    points_text = "\n\n".join(
        f"Claim: {p['claim']}\n"
        f"Source: {p['source_title']} ({p['source_type']})\n"
        f"URL: {p['source_url']}\n"
        f"Confidence: {p['confidence_score']:.2f} | Relevance: {p['relevance_score']:.2f}"
        for p in top_points
    )

    # Build a deduplicated source list for the report
    seen_urls = set()
    sources_list = []
    for p in top_points:
        url = p.get("source_url", "")
        if url and url not in seen_urls:
            seen_urls.add(url)
            year = ""
            sources_list.append(f"{p['source_title']} — {url}")

    result = writer_llm.invoke([
        SystemMessage(content="You are an expert academic writer producing a rigorous research report."),
        HumanMessage(content=(
            f'Write a comprehensive deep research report on: "{topic}"\n\n'
            f"User context (from clarifying questions):\n{qa_context}\n\n"
            f"Synthesized findings:\n{synthesis}\n\n"
            f"Top rated research points:\n{points_text}\n\n"
            "Report sections required:\n"
            "1. executive_summary — 3 thorough paragraphs for the stated audience\n"
            "2. key_facts — 8–12 highest-confidence facts as plain strings (no markdown)\n"
            "3. detailed_research — 600–900 word analysis with inline citations as [Source Title]\n"
            "4. sources — formatted references (use the list below exactly)\n"
            "5. further_reading — 5–7 recommendations: next searches, related topics, key authors\n\n"
            f"Sources to use: {sources_list}"
        ))
    ])

    report_dict = result.model_dump()
    report_dict["sources"] = sources_list  # use our curated, deduplicated list

    print("[Report Writer] Final report assembled.")
    return {"final_report": report_dict}

## Node 7 — Email Agent
Sends the formatted report by email. Reads the recipient address from state.
Silently skips if no address is provided or credentials are not configured.

In [ ]:
def email_node(state: State) -> dict:
    """Send the research report by email to the recipient in state."""
    report = state.get("final_report")
    recipient = state.get("recipient_email", "")
    topic = state.get("topic", "")

    if not report or not recipient:
        print("[Email Agent] Skipped — no report or no recipient.")
        return {"email_sent": False}

    report_md = format_report_markdown(report, topic)
    success = send_email_report(report_md, recipient, topic)
    return {"email_sent": success}

## Delivery
Final report is shown as markdown in the UI. Optional HTML email via SendGrid (`SENDGRID_API_KEY`, `EMAIL_FROM`).


In [ ]:
# PDF generation removed — use on-page markdown and SendGrid email only.


## Helper — Format Report as Markdown
Converts a `FinalReport` dict into a human-readable markdown string
used by the Gradio UI and the email agent.

In [ ]:
def format_report_markdown(report: dict, topic: str) -> str:
    """Render a FinalReport dict as a formatted markdown string."""
    lines = [
        f"# Research Report: {topic}",
        "",
        "## Executive Summary",
        report.get("executive_summary", ""),
        "",
        "## Key Facts",
        *[f"- {fact}" for fact in report.get("key_facts", [])],
        "",
        "## Detailed Research",
        report.get("detailed_research", ""),
        "",
        "## Sources & References",
        *[f"- {s}" for s in report.get("sources", [])],
        "",
        "## Further Reading",
        *[f"- {item}" for item in report.get("further_reading", [])],
    ]
    return "\n".join(lines)

---
## Graph Build Step 1 — Create the StateGraph

In [ ]:
# Initialise the StateGraph with our TypedDict State
graph_builder = StateGraph(State)

## Graph Build Step 2 — Register All Nodes

In [ ]:
# Register every node function with the graph builder
graph_builder.add_node("clarifier",     clarifier_node)
graph_builder.add_node("task_planner",  task_planner_node)
graph_builder.add_node("agent_discovery", agent_discovery)
graph_builder.add_node("agent_fulltext",  agent_fulltext)
graph_builder.add_node("agent_context",   agent_context)
graph_builder.add_node("agent_domain",    agent_domain)
graph_builder.add_node("synthesizer",   synthesizer_node)
graph_builder.add_node("rater",         rater_node)
graph_builder.add_node("report_writer", report_writer_node)
graph_builder.add_node("email_agent",   email_node)

print("All nodes registered.")


## Graph Build Step 3 — Wire the Edges

Key wiring decisions:
- `task_planner → fan_out_researchers` uses `add_conditional_edges` with `Send` (parallel fan-out)
- All 4 agents edge to `synthesizer` (fan-in — LangGraph waits for all to complete)
- `report_writer → email_agent → END` delivers optional SendGrid email after the report is written


In [ ]:
# Entry point
graph_builder.add_edge(START, "clarifier")
graph_builder.add_edge("clarifier", "task_planner")

# Fan-out: task_planner → 4 parallel research agents via Send API
graph_builder.add_conditional_edges("task_planner", fan_out_researchers)

# Fan-in: all parallel agents → synthesizer
graph_builder.add_edge("agent_discovery", "synthesizer")
graph_builder.add_edge("agent_fulltext",  "synthesizer")
graph_builder.add_edge("agent_context",   "synthesizer")
graph_builder.add_edge("agent_domain",    "synthesizer")

# Post-synthesis linear flow
graph_builder.add_edge("synthesizer",   "rater")
graph_builder.add_edge("rater",         "report_writer")
graph_builder.add_edge("report_writer", "email_agent")
graph_builder.add_edge("email_agent",   END)

print("All edges wired.")


## Graph Build Step 4 — Compile with Memory and Human-in-the-Loop Interrupt

`interrupt_after=["clarifier"]` — the clarifier runs and generates questions,
then the graph **pauses**. The Gradio UI displays the questions and waits for
user answers before calling `graph.update_state()` + `await asyncio.to_thread(graph.invoke, None, config)`
to resume.

In [ ]:
# Compile with SqliteSaver and interrupt_after clarifier for human approval gate
graph = graph_builder.compile(
    checkpointer=memory,
    interrupt_after=["clarifier"],
)

print("Graph compiled with interrupt_after=['clarifier'].")

## Visualise the Graph

In [ ]:
# Render the full graph as a Mermaid diagram
display(Image(graph.get_graph().draw_mermaid_png()))

---
## Gradio Helper — Generate a Unique Thread ID per Session

In [ ]:
def make_thread_id() -> str:
    """Generate a unique thread ID for each research session."""
    return str(uuid.uuid4())

## Gradio Handler — Phase 1: Submit Topic and Receive Clarifying Questions

Starts the graph, runs the clarifier, hits the interrupt, and returns
the generated questions for display in the UI.

In [ ]:

from langsmith.run_helpers import tracing_context
from langsmith.run_trees import RunTree
from langsmith.utils import tracing_is_enabled

LS_SESSION_ROOT_RUNS: dict[str, RunTree] = {}
LS_GRAPH_RUN_NAME = "AcademicDeepResearch"


def research_graph_config(thread_id: str) -> dict:
    return {"configurable": {"thread_id": thread_id}, "run_name": LS_GRAPH_RUN_NAME}


def _abandon_session_root_run(thread_id: str, *, status: str) -> None:
    root = LS_SESSION_ROOT_RUNS.pop(thread_id, None)
    if root is None:
        return
    root.end(outputs={"status": status})
    try:
        root.patch()
    except Exception:
        pass


def begin_session_root_run(thread_id: str, topic: str) -> RunTree | None:
    if tracing_is_enabled() is not True:
        return None
    _abandon_session_root_run(thread_id, status="superseded")
    root = RunTree(
        name="Academic deep research session",
        run_type="chain",
        inputs={"thread_id": thread_id, "topic": topic},
        tags=["academic-deep-researcher"],
    )
    root.add_metadata({"thread_id": thread_id, "langgraph_node": "session"})
    root.post()
    LS_SESSION_ROOT_RUNS[thread_id] = root
    return root


def finish_session_root_run(thread_id: str, *, error: str | None = None) -> None:
    root = LS_SESSION_ROOT_RUNS.pop(thread_id, None)
    if root is None:
        return
    if error:
        root.end(error=error)
    else:
        root.end(outputs={"status": "completed"})
    try:
        root.patch()
    except Exception:
        pass


async def start_research(topic: str, thread_id: str, chat_history: list):
    """Run the graph until interrupt_after clarifier; update chat with topic + questions."""
    msgs = list(chat_history or [])

    if not topic.strip():
        msgs.append({"role": "assistant", "content": "Please enter a research topic."})
        yield msgs, gr.update(visible=False), thread_id
        return

    msgs.append({"role": "user", "content": topic.strip()})
    msgs.append({"role": "assistant", "content": "_Generating clarifying questions…_"})
    yield msgs, gr.update(visible=False), thread_id

    config = research_graph_config(thread_id)
    root_run = begin_session_root_run(thread_id, topic.strip())

    initial_state = {
        "topic": topic.strip(),
        "messages": [HumanMessage(content=topic.strip())],
        "clarifying_questions": [],
        "clarifying_qa": [],
        "clarifications_approved": False,
        "refined_topic": "",
        "raw_results": [],
        "rated_points": [],
        "synthesized_findings": "",
        "research_tasks": [],
        "agent_queries": None,
        "final_report": None,
        "email_sent": False,
        "recipient_email": "",
    }

    def _invoke_clarify():
        if root_run is not None:
            with tracing_context(parent=root_run):
                return graph.invoke(initial_state, config)
        return graph.invoke(initial_state, config)

    await asyncio.to_thread(_invoke_clarify)

    snapshot = graph.get_state(config)
    questions = snapshot.values.get("clarifying_questions", [])

    if not questions:
        finish_session_root_run(thread_id, error="clarifier produced no questions")
        msgs[-1] = {"role": "assistant", "content": "No questions generated. Check OpenAI key."}
        yield msgs, gr.update(visible=False), thread_id
        return

    questions_md = (
        "**Please answer each question below (one answer per line):**\n\n"
        + "\n".join(f"**{i+1}.** {q}" for i, q in enumerate(questions))
    )
    msgs[-1] = {"role": "assistant", "content": questions_md}
    yield msgs, gr.update(visible=True), thread_id


## Gradio Handler — Phase 2: Approve Answers and Run Full Research Pipeline

Updates checkpoint with answers, then streams each node completion into the chat using **`graph.stream(..., stream_mode="updates")`** on a **background thread** (required because **`SqliteSaver` does not implement async checkpointing**, so **`graph.astream` raises `NotImplementedError`**). The refined question appears after **task_planner**. Final markdown report and email status fill the panels below.


In [ ]:
_STREAM_DONE = object()


async def _stream_graph_updates(graph, config, parent_run=None):
    """Bridge sync `graph.stream` into async for Gradio. SqliteSaver has no `aput`/`aget_tuple`; `astream` fails."""
    loop = asyncio.get_running_loop()
    q: asyncio.Queue = asyncio.Queue()
    err: list[BaseException] = []

    def worker():
        try:
            def iter_chunks():
                if parent_run is not None:
                    with tracing_context(parent=parent_run):
                        yield from graph.stream(None, config, stream_mode="updates")
                else:
                    yield from graph.stream(None, config, stream_mode="updates")

            for chunk in iter_chunks():
                asyncio.run_coroutine_threadsafe(q.put(chunk), loop).result()
        except BaseException as e:
            err.append(e)
        finally:
            asyncio.run_coroutine_threadsafe(q.put(_STREAM_DONE), loop).result()

    ctx = contextvars.copy_context()
    threading.Thread(target=lambda: ctx.run(worker), daemon=True).start()
    while True:
        chunk = await q.get()
        if chunk is _STREAM_DONE:
            if err:
                raise err[0]
            return
        yield chunk


def _node_progress_message(node: str, upd: dict) -> str | None:
    if node == "task_planner":
        return None
    if node in ("agent_discovery", "agent_fulltext", "agent_context", "agent_domain"):
        n = len(upd.get("raw_results") or [])
        meta = {
            "agent_discovery": ("Semantic Scholar", "Paper metadata and citations"),
            "agent_fulltext": ("ArXiv & CORE", "Preprints and open-access works"),
            "agent_context": ("Web (Tavily)", "News and practical context"),
            "agent_domain": ("PubMed", "Biomedical literature (when enabled)"),
        }
        title, subtitle = meta[node]
        if n == 0:
            hints = {
                "agent_discovery": "No Semantic Scholar data (optional source). Set **SEMANTIC_SCHOLAR_API_KEY**, or **SEMANTIC_SCHOLAR_ALLOW_ANONYMOUS=true** for the public API (strict quotas / 429 retries).",
                "agent_fulltext": "Nothing matched or ArXiv/CORE throttled. ArXiv allows ~1 request per 3s; CORE may return 5xx — retries are built in.",
                "agent_context": "No web snippets for this query.",
                "agent_domain": "No PubMed run (non-biomedical topic) or no hits.",
            }
            extra = hints[node]
            body = f"**0** sources  \n_{extra}_"
        else:
            body = f"**{n}** source(s) will feed the synthesis."
        return f"### {title}\n*{subtitle}*\n\n{body}\n"
    if node == "synthesizer":
        return "### Synthesis\nCombined all retrieved snippets into one working set.\n"
    if node == "rater":
        n = len(upd.get("rated_points") or [])
        return f"### Quality check\n**{n}** claim(s) passed confidence and relevance thresholds.\n"
    if node == "report_writer":
        return "### Report draft\nStructured sections and citations assembled.\n"
    if node == "email_agent":
        if upd.get("email_sent"):
            return "### Email\nHTML report sent via **SendGrid**.\n"
        return "### Email\nNot sent (missing recipient or SendGrid config).\n"
    if node.startswith("__"):
        return None
    return f"### Update\nFinished step `{node}`.\n"


async def run_research(answers_text: str, email: str, thread_id: str, chat_history: list):
    """Resume the graph; stream node updates into the chat; fill report + email status."""
    answers_text = answers_text or ""
    email = email or ""
    msgs = list(chat_history or [])
    config = research_graph_config(thread_id)
    root_run = LS_SESSION_ROOT_RUNS.get(thread_id)
    err_note: str | None = None
    try:
        snapshot = graph.get_state(config)
        questions = snapshot.values.get("clarifying_questions", [])

        answer_lines = [a.strip() for a in answers_text.strip().split("\n") if a.strip()]
        qa_list = [
            {"question": q, "answer": answer_lines[i] if i < len(answer_lines) else "Not specified"}
            for i, q in enumerate(questions)
        ]

        user_summary = "Submitted answers to the clarifying questions."
        if answer_lines:
            user_summary += "\n\n" + "\n".join(answer_lines[:12])
        msgs.append({"role": "user", "content": user_summary})
        yield msgs, "", ""

        if root_run is not None:
            with tracing_context(parent=root_run):
                graph.update_state(
                    config,
                    {
                        "clarifying_qa": qa_list,
                        "clarifications_approved": True,
                        "recipient_email": email.strip(),
                        "messages": [HumanMessage(content=user_summary)],
                    },
                )
        else:
            graph.update_state(
                config,
                {
                    "clarifying_qa": qa_list,
                    "clarifications_approved": True,
                    "recipient_email": email.strip(),
                    "messages": [HumanMessage(content=user_summary)],
                },
            )

        async for chunk in _stream_graph_updates(graph, config, root_run):
            if not isinstance(chunk, dict):
                continue
            for node, upd in chunk.items():
                if node == "task_planner" and isinstance(upd, dict):
                    refined = (upd.get("refined_topic") or "").strip()
                    tasks = upd.get("research_tasks") or []
                    aq = upd.get("agent_queries") or {}
                    needs = aq.get("needs_pubmed", False) if isinstance(aq, dict) else False
                    pub = "**on**" if needs else "**off**"
                    plan_line = f"Planned **{len(tasks)}** research sub-tasks · PubMed branch {pub}."
                    if refined:
                        text = f"### Refined research question\n\n{refined}\n\n---\n\n### Plan\n{plan_line}\n"
                    else:
                        text = f"### Plan\n{plan_line}\n"
                    msgs.append({"role": "assistant", "content": text})
                else:
                    line = _node_progress_message(node, upd if isinstance(upd, dict) else {})
                    if line:
                        msgs.append({"role": "assistant", "content": line})
                yield msgs, "", ""

        snap = graph.get_state(config)
        result = snap.values
        report = result.get("final_report")
        email_sent = result.get("email_sent", False)

        if not report:
            msgs.append({"role": "assistant", "content": "### Result\nResearch finished but no report was produced.\n"})
            yield msgs, "", ""
            err_note = "no final_report"
            return

        topic = result.get("topic", "")
        report_md = format_report_markdown(report, topic)
        email_status = f"Report emailed to {email.strip()}" if email_sent else ""
        if not email.strip():
            email_status = "No email sent (no address provided)."
        elif not email_sent:
            email_status = "Email not sent (check SendGrid / EMAIL_FROM)."

        msgs.append({"role": "assistant", "content": "### Done\nFull **markdown report** is in the panel below.\n"})
        yield msgs, report_md, email_status
    except BaseException as e:
        err_note = str(e)
        raise
    finally:
        if thread_id in LS_SESSION_ROOT_RUNS:
            finish_session_root_run(thread_id, error=err_note)



---
## Launch — Academic Deep Researcher UI

Two-phase Gradio interface with a **chat** timeline:
- **Phase 1:** Enter topic → clarifier runs → questions appear in the chat
- **Phase 2:** Answer questions → pipeline streams progress in chat → markdown report + email status


In [ ]:
with gr.Blocks(theme=gr.themes.Default(primary_hue="emerald")) as demo:
    gr.Markdown("# Academic Multi-Agent Deep Researcher")
    _ls_on = os.getenv("LANGSMITH_TRACING", "").lower() in ("true", "1", "yes")
    _ls_key = (os.getenv("LANGSMITH_API_KEY") or "").strip()
    _ls_proj = os.getenv("LANGSMITH_PROJECT") or os.getenv("LANGCHAIN_PROJECT", "academic-deep-researcher")
    gr.Markdown(
        "**[LangSmith](https://smith.langchain.com)** · "
        + (
            f"tracing **on** — project `{_ls_proj}`"
            if (_ls_on and _ls_key)
            else "tracing **off** — set `LANGSMITH_TRACING=true` and `LANGSMITH_API_KEY` to record runs"
        )
    )
    gr.Markdown(
        "Powered by Semantic Scholar · ArXiv · CORE · Tavily · PubMed (conditional)\n\n"
        "Use the chat for your topic, clarifying questions, and live research progress. "
        "Answer the questions in Step 2, then run deep research."
    )

    thread_id = gr.State(make_thread_id)
    chatbot = gr.Chatbot(label="Research chat", type="messages", height=420)

    with gr.Group():
        gr.Markdown("### Step 1 — Enter your research topic")
        topic_input = gr.Textbox(
            label="Research Topic",
            placeholder="e.g. The impact of transformer architecture on natural language processing",
        )
        start_btn = gr.Button("Generate Clarifying Questions", variant="primary")

    with gr.Group(visible=False) as phase2_group:
        gr.Markdown("### Step 2 — Answer the clarifying questions (same order as in chat)")
        answers_input = gr.Textbox(
            label="Your Answers (one per line, matching question order)",
            lines=5,
            placeholder="Answer 1\nAnswer 2\nAnswer 3\nAnswer 4",
        )
        email_input = gr.Textbox(
            label="Email address for report delivery (optional)",
            placeholder="your@email.com",
        )
        research_btn = gr.Button("Approve & Start Deep Research", variant="primary")

    with gr.Group():
        gr.Markdown("### Research Report (markdown)")
        report_output = gr.Markdown()
        email_status = gr.Markdown()

    start_btn.click(
        fn=start_research,
        inputs=[topic_input, thread_id, chatbot],
        outputs=[chatbot, phase2_group, thread_id],
    )

    research_btn.click(
        fn=run_research,
        inputs=[answers_input, email_input, thread_id, chatbot],
        outputs=[chatbot, report_output, email_status],
    )

demo.launch(inbrowser=True)


[Semantic Scholar] 429 — backing off 17.7s (attempt 5/5)
[Semantic Scholar] Error after retries: None
[Discovery Agent] 0 papers from Semantic Scholar
